[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/01_retail-sales-analysis.ipynb)

*Industry Project 1 of 10 · Data Analyst · Beginner–Intermediate · about 6–8 hours*

# Retail Sales Analysis & Forecast

**The situation.** You've joined the analytics team of *ShopKart*, an online retailer selling electronics, home goods, fashion, groceries and beauty products across four regions. The head of sales has a quarterly review next week and asks you:

1. How big is the business right now, and is it growing?
2. Which regions and customer segments bring in the money?
3. How dependent are we on a few big customers?
4. Do new customers come back the next month?
5. What revenue should we plan for over the next three months?

The raw export from the order system is messy: it has duplicate rows and returned orders mixed in.

**What you'll hand in:** a cleaned dataset, a KPI summary, a regional breakdown from SQL, a Pareto and retention analysis, a 3-month forecast with the method you chose and why, and a one-page summary for the head of sales.

**Skills:** pandas, data cleaning, KPIs, SQL (sqlite3), Pareto analysis, cohort retention, forecasting, storytelling

**How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

Two files, 18 months of orders (Jan 2025 – Jun 2026):

**retail_orders.csv** — one row per order line

| Column | Meaning |
|---|---|
| order_id | order number (the export has some **duplicate rows**) |
| order_date | date, as text `YYYY-MM-DD` |
| customer_id | who ordered |
| category | product category |
| units, unit_price | quantity and price per unit (₹) |
| discount | share taken off, e.g. 0.1 = 10% |
| returned | 1 if the order was returned and refunded |

**retail_customers.csv** — customer_id, segment (Consumer / Corporate / Small Business), region, signup_date

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return
    print(f"❌ {name}: not yet — {_why(got, want)}")

import sqlite3
import matplotlib.pyplot as plt
orders = load("retail_orders.csv")
customers = load("retail_customers.csv")
print(orders.shape, customers.shape)
print("Ready! Data loaded.")

Take a first look before you start. What does one row mean? Which columns need fixing?

In [ ]:
display(orders.head())
print(orders.dtypes)
print('duplicate order ids:', orders.order_id.duplicated().sum())

## Milestone 1: Clean the orders

Write `clean(orders)` that returns a **new** DataFrame where:
- duplicate rows are removed (keep the first row for each `order_id`),
- `order_date` is a real datetime,
- a `revenue` column = units × unit_price × (1 − discount), rounded to 2 decimals,
- **returned orders are removed** (they were refunded, so they aren't revenue).

The check compares the number of rows and the total revenue.

<details>
<summary><b>💡 Hint</b></summary>

`drop_duplicates(subset='order_id')`, `pd.to_datetime`, then filter with `df[df.returned == 0]`. Work on a copy so you don't change `orders`.

</details>

In [ ]:
def clean(orders):
    df = orders.copy()
    # 1. remove duplicate order_ids
    # 2. convert order_date
    # 3. add revenue
    # 4. drop returned orders
    raise NotImplementedError

In [ ]:
check("Clean the orders", lambda: (len(clean(orders)), round(clean(orders)['revenue'].sum(), 2)), [6016, 8358929.13], tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def clean(orders):
    df = orders.drop_duplicates(subset="order_id").copy()
    df["order_date"] = pd.to_datetime(df["order_date"])
    df["revenue"] = (df["units"] * df["unit_price"] * (1 - df["discount"])).round(2)
    return df[df["returned"] == 0].reset_index(drop=True)
```

</details>

## Milestone 2: Headline KPIs

Write `kpis(orders)` returning a dict with:
- `revenue`: total revenue of clean orders (rounded to 2)
- `orders`: number of clean orders
- `customers`: number of distinct customers who bought
- `aov`: average order value = revenue ÷ orders (rounded to 2)
- `return_rate_pct`: % of **de-duplicated** orders that were returned (rounded to 2)

<details>
<summary><b>💡 Hint</b></summary>

Use `clean()` for the first four. For the return rate you need the de-duplicated orders *before* returns were removed.

</details>

In [ ]:
def kpis(orders):
    sales = clean(orders)
    raise NotImplementedError

In [ ]:
check("Headline KPIs", lambda: kpis(orders), {'revenue': 8358929.13, 'orders': 6016, 'customers': 533, 'aov': 1389.45, 'return_rate_pct': 4.26}, tol=0.001)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def kpis(orders):
    sales = clean(orders)
    dedup = orders.drop_duplicates(subset="order_id")
    revenue = sales["revenue"].sum()
    return {"revenue": round(revenue, 2), "orders": len(sales), "customers": sales["customer_id"].nunique(),
            "aov": round(revenue / len(sales), 2), "return_rate_pct": round(dedup["returned"].mean() * 100, 2)}
```

</details>

## Milestone 3: Monthly revenue and growth

Write `monthly(sales)` that takes the **clean** data and returns a Series of revenue per month, indexed by text like `'2025-01'`, rounded to 2. Then look at the growth: which months are seasonal peaks?

<details>
<summary><b>💡 Hint</b></summary>

`sales.order_date.dt.strftime('%Y-%m')` gives the month text; group by it and sum.

</details>

In [ ]:
def monthly(sales):
    raise NotImplementedError

In [ ]:
check("Monthly revenue and growth", lambda: monthly(clean(orders)), {'2025-01': 412630.94, '2025-02': 266193.98, '2025-03': 384858.68, '2025-04': 396353.22, '2025-05': 418948.01, '2025-06': 325102.6, '2025-07': 368341.76, '2025-08': 436026.79, '2025-09': 403652.16, '2025-10': 662263.99, '2025-11': 696247.63, '2025-12': 878601.61, '2026-01': 420964.86, '2026-02': 361354.28, '2026-03': 485785.18, '2026-04': 524236.24, '2026-05': 492621.36, '2026-06': 424745.84}, tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def monthly(sales):
    month = sales["order_date"].dt.strftime("%Y-%m")
    return sales.groupby(month)["revenue"].sum().round(2)
```

</details>

Plot it: `monthly(clean(orders)).plot(kind='bar', figsize=(10, 3))`. Then compute month-over-month growth with `.pct_change()`. Is the business growing once you look past the October–December peak?

## Milestone 4: Revenue by region and segment — in SQL

Managers will want this from the company database, so do it in SQL. Write `top_groups(sales, customers)`:
1. load both tables into an in-memory SQLite database (`df.to_sql`),
2. join sales to customers,
3. return the **top 5** `(region, segment, revenue)` rows by revenue, with revenue rounded to 2 **in SQL**.

<details>
<summary><b>💡 Hint</b></summary>

Store `order_date` as text before `to_sql` (or select only the columns you need). `SELECT c.region, c.segment, ROUND(SUM(s.revenue), 2) AS rev FROM sales s JOIN customers c ON ... GROUP BY ... ORDER BY rev DESC LIMIT 5`.

</details>

In [ ]:
def top_groups(sales, customers):
    con = sqlite3.connect(":memory:")
    raise NotImplementedError

In [ ]:
check("Revenue by region and segment \u2014 in SQL", lambda: top_groups(clean(orders), customers), [['South', 'Corporate', 1533234.39], ['North', 'Consumer', 1098887.84], ['West', 'Consumer', 918033.59], ['West', 'Corporate', 889578.66], ['South', 'Consumer', 887883.26]], tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def top_groups(sales, customers):
    con = sqlite3.connect(":memory:")
    sales[["order_id", "customer_id", "revenue"]].to_sql("sales", con, index=False)
    customers.to_sql("customers", con, index=False)
    sql = '''SELECT c.region, c.segment, ROUND(SUM(s.revenue), 2) AS rev
             FROM sales s JOIN customers c ON c.customer_id = s.customer_id
             GROUP BY c.region, c.segment ORDER BY rev DESC LIMIT 5'''
    return con.execute(sql).fetchall()
```

</details>

## Milestone 5: How concentrated is revenue?

Write `pareto(sales, pct)` that returns the % of revenue (rounded to 1) that comes from the top `pct`% of customers by spend (round the number of customers **up**). Run it for 20% — is it close to the famous 80/20?

<details>
<summary><b>💡 Hint</b></summary>

Total per customer with groupby, `math.ceil(n * pct / 100)` customers, `nlargest`.

</details>

In [ ]:
def pareto(sales, pct):
    raise NotImplementedError

In [ ]:
check("How concentrated is revenue?", lambda: [pareto(clean(orders), 20), pareto(clean(orders), 5)], [66.2, 36.2], tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def pareto(sales, pct):
    spend = sales.groupby("customer_id")["revenue"].sum()
    k = math.ceil(len(spend) * pct / 100)
    return round(spend.nlargest(k).sum() / spend.sum() * 100, 1)
```

</details>

## Milestone 6: Do new customers come back?

A customer's **cohort** is the month of their first purchase. Month-1 retention is the % of a cohort that buys again in the **next** calendar month.

Write `month1_retention(sales)` returning a dict `{cohort: retention %}` (rounded to 1) for every cohort except the last month in the data (it has no next month yet).

<details>
<summary><b>💡 Hint</b></summary>

Turn dates into monthly periods (`dt.to_period('M')`). First period per customer with `groupby().transform('min')`. A customer is retained if they have any order in cohort + 1.

</details>

In [ ]:
def month1_retention(sales):
    raise NotImplementedError

In [ ]:
check("Do new customers come back?", lambda: month1_retention(clean(orders)), {'2025-01': 40.6, '2025-02': 33.9, '2025-03': 34.0, '2025-04': 21.1, '2025-05': 16.7, '2025-06': 20.8, '2025-07': 22.7, '2025-08': 11.5, '2025-09': 20.8, '2025-10': 13.3, '2025-11': 8.3, '2025-12': 14.3, '2026-01': 20.0, '2026-02': 0.0, '2026-03': 11.1, '2026-04': 33.3, '2026-05': 20.0}, tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def month1_retention(sales):
    p = sales["order_date"].dt.to_period("M")
    cohort = p.groupby(sales["customer_id"]).transform("min")
    active = set(zip(sales["customer_id"], p))
    first = pd.DataFrame({"customer_id": sales["customer_id"], "cohort": cohort}).drop_duplicates()
    out = {}
    for c, group in first.groupby("cohort"):
        if c == p.max():
            continue
        back = sum((cid, c + 1) in active for cid in group["customer_id"])
        out[str(c)] = round(back / len(group) * 100, 1)
    return out
```

</details>

Cohorts get small later on (most customers have already bought once), so later percentages are noisier. Say that in your report rather than over-reading them.

## Milestone 7: Forecast the next quarter

Write `forecast(sales)`:
1. Build monthly revenue (your `monthly`).
2. **Backtest**: hold out the last 3 months, forecast them from the rest with three methods, and compute each one's MAE:
   - `naive`: the last training month repeated,
   - `mean3`: the mean of the last 3 training months,
   - `seasonal`: the same month one year earlier (value 12 months back).
3. Pick the method with the lowest MAE, then use it on **all** the data to forecast the next 3 months.

Return `(best_method, [3 forecasts rounded to 0])`.

<details>
<summary><b>💡 Hint</b></summary>

With values `v` (a list) and training part `t = v[:-3]`: seasonal forecast for step i is `t[len(t) - 12 + i]`. For the final forecast use the full series the same way.

</details>

In [ ]:
def forecast(sales):
    v = monthly(sales).tolist()
    raise NotImplementedError

In [ ]:
check("Forecast the next quarter", lambda: forecast(clean(orders)), ['naive', [424746, 424746, 424746]], tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def forecast(sales):
    v = monthly(sales).tolist()

    def predict(t, method):
        if method == "naive":
            return [t[-1]] * 3
        if method == "mean3":
            return [sum(t[-3:]) / 3] * 3
        return [t[len(t) - 12 + i] for i in range(3)]

    train, test = v[:-3], v[-3:]
    mae = {m: sum(abs(a - b) for a, b in zip(test, predict(train, m))) / 3 for m in ["naive", "mean3", "seasonal"]}
    best = min(mae, key=mae.get)
    return best, [round(x) for x in predict(v, best)]
```

</details>

A forecast without an error range is a guess. Report the backtest MAE next to it, e.g. *"₹X ± MAE per month"*.

## A chart for the slide

One message per chart, stated in the title. Change the title once you know your headline.

In [ ]:
rev = load("retail_orders.csv").drop_duplicates("order_id")
rev = rev[rev.returned == 0].assign(revenue=lambda d: d.units * d.unit_price * (1 - d.discount))
by_month = rev.groupby(rev.order_date.str[:7])["revenue"].sum() / 1e5
ax = by_month.plot(kind="bar", figsize=(10, 3), color="#0f766e")
ax.set_ylabel("revenue (₹ lakh)")
ax.set_title("Revenue by month: peaks every Oct–Dec")
plt.show()

## Present your work

Write a one-page summary for the head of sales (in a markdown cell, or a short slide deck):

1. **Headline** in one sentence: size and growth of the business.
2. **Where the money is**: top regions/segments, and the Pareto share — what it means for risk.
3. **Retention**: what share of new customers come back, and whether it's improving.
4. **Next quarter**: the forecast, the method, and its typical error.
5. **Recommendations** (2–3), each tied to a number above — e.g. a win-back campaign for one-time buyers.
6. **Caveats**: returns removed, duplicates removed, small late cohorts.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Cleaned and analysed 18 months of e-commerce orders (≈10k rows) with pandas and SQL, removing duplicate and refunded orders
- Built KPI, regional and cohort-retention reports, finding the top 20% of customers drive **X%** of revenue
- Backtested naive, moving-average and seasonal forecasts; the chosen method's error was **₹Y** per month

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. How did you decide what counts as revenue? What happens to the KPIs if you forget the duplicates or returns?
2. Why did you choose that forecasting method, and how would you know if it stops working?
3. Your top 20% of customers bring most of the revenue. What would you recommend, and how would you measure it?
4. Month-1 retention dropped for later cohorts. Is that a real problem? How would you check?
5. How would this analysis change if the data were 100 million rows in a warehouse?

## Take it further

- Add a returns analysis: which categories are returned most, and how much revenue does that cost?
- Build an interactive dashboard of the KPIs (Streamlit, Power BI or Tableau) from your clean data.
- Forecast per category instead of in total, and compare the sum with the total forecast.
- Segment customers with RFM (recency, frequency, monetary) scores.